# 04d - Model Comparison

**Objective**: Compare all forecasting models and select the best

In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
print('Libraries imported')

Libraries imported


In [13]:
# Load all forecasts
prophet_fc = pd.read_csv('../models/trained/prophet_forecast.csv', parse_dates=['ds'])
sarima_fc = pd.read_csv('../models/trained/sarima_forecast.csv', parse_dates=['date'])
xgb_fc = pd.read_csv('../models/trained/xgboost_forecast.csv', parse_dates=['date'])
try:
    actual = pd.read_csv('../data/clean/maize_features.csv')
except FileNotFoundError:
    actual = pd.read_csv('data/clean/maize_features.csv')
print('✓ Forecasts loaded')

✓ Forecasts loaded


In [14]:
# Create comparison dataframe
test_actual = actual.iloc[-12:][['date', 'price']].copy()
test_actual = test_actual.reset_index(drop=True)

prophet_test = prophet_fc.tail(12)[['ds', 'yhat']].reset_index(drop=True)
prophet_test.columns = ['date', 'prophet_forecast']

# Reload sarima_fc and handle the numeric dates
sarima_fc = pd.read_csv('../models/trained/sarima_forecast.csv')

# If sarima_fc has numeric dates, we need to get the actual dates from the original data
if sarima_fc['date'].dtype in ['int64', 'float64']:
    # Get the actual dates from the original data
    actual_dates = actual['date'].iloc[-12:].values
    sarima_fc['date'] = actual_dates

# Convert all date columns to datetime
test_actual['date'] = pd.to_datetime(test_actual['date'])
prophet_test['date'] = pd.to_datetime(prophet_test['date'])
sarima_fc['date'] = pd.to_datetime(sarima_fc['date'])
xgb_fc['date'] = pd.to_datetime(xgb_fc['date'])

# Now merge
comparison = test_actual.merge(prophet_test, on='date')
comparison = comparison.merge(sarima_fc[['date', 'forecast']].rename(columns={'forecast': 'sarima_forecast'}), on='date')
comparison = comparison.merge(xgb_fc[['date', 'forecast']].rename(columns={'forecast': 'xgb_forecast'}), on='date')
print(comparison)

         date      price  prophet_forecast  sarima_forecast  xgb_forecast
0  2020-01-01  54.650000         54.020855        58.977162     54.500393
1  2020-02-01  53.833333         53.003730        59.789574     55.081180
2  2020-03-01  47.900000         52.734765        60.484518     48.221850
3  2020-04-01  58.037500         51.051003        61.008729     53.176800
4  2020-05-01  57.533333         55.333136        62.025239     55.411156
5  2020-06-01  55.966667         55.429203        62.040791     55.923120
6  2020-07-01  55.477778         55.277846        60.517774     56.391580
7  2020-08-01  55.837500         53.972588        57.994659     55.980087
8  2020-09-01  54.233333         52.797382        57.276100     54.321037
9  2020-10-01  53.988889         53.964598        56.328823     55.021130
10 2020-11-01  52.755556         54.020682        56.075214     50.536022
11 2020-12-01  52.733333         53.172705        55.111685     51.874607


In [15]:
# Calculate metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error
metrics = {}
for model in ['prophet_forecast', 'sarima_forecast', 'xgb_forecast']:
    mae = mean_absolute_error(comparison['price'], comparison[model])
    rmse = np.sqrt(mean_squared_error(comparison['price'], comparison[model]))
    mape = np.mean(np.abs((comparison['price'] - comparison[model]) / comparison['price'])) * 100
    metrics[model] = {'MAE': mae, 'RMSE': rmse, 'MAPE': mape}
metrics_df = pd.DataFrame(metrics).T
print('\n=== MODEL COMPARISON ===')
print(metrics_df.round(2))


=== MODEL COMPARISON ===
                   MAE  RMSE  MAPE
prophet_forecast  1.77  2.67  3.27
sarima_forecast   4.56  5.32  8.56
xgb_forecast      1.17  1.77  2.10


In [16]:
from pathlib import Path
Path('../models/trained').mkdir(parents=True, exist_ok=True)

# Save comparison
metrics_df.to_csv('../models/trained/model_comparison.csv')
comparison.to_csv('../models/trained/forecasts_comparison.csv', index=False)
print('Comparison saved')


Comparison saved


In [17]:
# Best model
best_model = metrics_df['MAPE'].idxmin()
print(f'\n🏆 Best Model: {best_model}')
print(f'MAPE: {metrics_df.loc[best_model, "MAPE"]:.2f}%')


🏆 Best Model: xgb_forecast
MAPE: 2.10%
